In [4]:
import pandas as pd
from sqlalchemy import create_engine

# ============================================
# НАСТРОЙКИ ПОДКЛЮЧЕНИЯ (ЗАМЕНИ НА СВОИ)
# ============================================
DB_NAME = 'ujin901'
DB_USER = 'postgres'
DB_PASSWORD = '16911691'  # ← ЗАМЕНИ НА РЕАЛЬНЫЙ ПАРОЛЬ
DB_HOST = 'localhost'
DB_PORT = '5433'

# Создаем подключение
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# Проверка подключения
try:
    with engine.connect() as conn:
        print(" Подключение к PostgreSQL успешно")
except Exception as e:
    print(f" Ошибка подключения: {e}")

# Загружаем агрегированные данные
query = """
SELECT 
    group_name,
    COUNT(DISTINCT user_id) as users,
    SUM(purchase_flag) as purchases
FROM pen_shop_ab_test
GROUP BY group_name
ORDER BY group_name
"""

df = pd.read_sql(query, engine)

print(" ДАННЫЕ ИЗ POSTGRESQL:")
print(df.to_string(index=False))

 Подключение к PostgreSQL успешно
 ДАННЫЕ ИЗ POSTGRESQL:
group_name  users  purchases
         A    676         37
         B    674         62


In [5]:
# Извлекаем данные для группы A
users_A = int(df[df['group_name'] == 'A']['users'].values[0])
purchases_A = int(df[df['group_name'] == 'A']['purchases'].values[0])

# Извлекаем данные для группы B
users_B = int(df[df['group_name'] == 'B']['users'].values[0])
purchases_B = int(df[df['group_name'] == 'B']['purchases'].values[0])

# Рассчитываем конверсию
conv_A = purchases_A / users_A
conv_B = purchases_B / users_B

print("=" * 50)
print("ИСХОДНЫЕ ДАННЫЕ ДЛЯ РАСЧЕТА")
print("=" * 50)
print(f"Группа A: {users_A} пользователей, {purchases_A} покупок, конверсия {conv_A:.4f} ({conv_A:.2%})")
print(f"Группа B: {users_B} пользователей, {purchases_B} покупок, конверсия {conv_B:.4f} ({conv_B:.2%})")
print(f"Разница: {(conv_B - conv_A):.4f} ({(conv_B - conv_A):.2%})")
print(f"Относительный прирост: {((conv_B - conv_A) / conv_A):.2%}")

ИСХОДНЫЕ ДАННЫЕ ДЛЯ РАСЧЕТА
Группа A: 676 пользователей, 37 покупок, конверсия 0.0547 (5.47%)
Группа B: 674 пользователей, 62 покупок, конверсия 0.0920 (9.20%)
Разница: 0.0373 (3.73%)
Относительный прирост: 68.06%


In [7]:
from statsmodels.stats.proportion import proportions_ztest

# Подготовка данных для z-test
count = [purchases_A, purchases_B]  # количество успехов
nobs = [users_A, users_B]           # размер выборки

# Двусторонний тест (проверяем, отличается ли B от A в любую сторону)
z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print("\n" + "=" * 50)
print("РЕЗУЛЬТАТЫ Z-TEST")
print("=" * 50)

print(f"p-value: {p_value:.6f}")


РЕЗУЛЬТАТЫ Z-TEST
p-value: 0.008654


In [8]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_lower, ci_upper = confint_proportions_2indep(
    count1=purchases_B,   # ← группа B (первая)
    nobs1=users_B,
    count2=purchases_A,   # ← группа A (вторая)
    nobs2=users_A,
    alpha=0.05,           # 95% доверительный интервал
    method='agresti-caffo'
)

print("\n" + "=" * 50)
print("95% ДОВЕРИТЕЛЬНЫЙ ИНТЕРВАЛ (для разницы B - A)")
print("=" * 50)
print(f"Нижняя граница: {ci_lower:.4f} ({ci_lower:.2%})")
print(f"Верхняя граница: {ci_upper:.4f} ({ci_upper:.2%})")
print(f"Точечная оценка: {(conv_B - conv_A):.4f} ({(conv_B - conv_A):.2%})")

# Проверка
if ci_lower > 0:
    print(" Интервал полностью ВЫШЕ 0 → разница положительная и значимая")
elif ci_upper < 0:
    print(" Интервал полностью НИЖЕ 0 → группа B хуже группы A")
else:
    print(" Интервал пересекает 0 → разница статистически НЕ значима")


95% ДОВЕРИТЕЛЬНЫЙ ИНТЕРВАЛ (для разницы B - A)
Нижняя граница: 0.0092 (0.92%)
Верхняя граница: 0.0651 (6.51%)
Точечная оценка: 0.0373 (3.73%)
 Интервал полностью ВЫШЕ 0 → разница положительная и значимая


In [9]:
from statsmodels.stats.power import zt_ind_solve_power
import numpy as np

# Параметры
alpha = 0.05      # уровень значимости
power = 0.80      # мощность теста (80%)
ratio = users_B / users_A  # соотношение групп

# Расчет MDE (стандартизированный)
mde_standardized = zt_ind_solve_power(
    effect_size=None,
    nobs1=users_A,
    alpha=alpha,
    power=power,
    ratio=ratio,
    alternative='two-sided'
)

# Конвертируем в абсолютные процентные пункты
pooled_prop = (purchases_A + purchases_B) / (users_A + users_B)
mde_absolute = mde_standardized * np.sqrt(pooled_prop * (1 - pooled_prop))

print("\n" + "=" * 50)
print("РАСЧЕТ MDE (Minimal Detectable Effect)")
print("=" * 50)
print(f"Уровень значимости (α): {alpha}")
print(f"Мощность теста (1-β): {power}")
print(f"Текущий размер выборки: A={users_A}, B={users_B}")
print(f"\nMDE (стандартизированный): {mde_standardized:.4f}")
print(f"MDE (в абсолютных п.п.): ±{mde_absolute:.4f} (±{mde_absolute:.2%})")
print(f"\nТекущий наблюдаемый эффект: {(conv_B - conv_A):.4f} ({(conv_B - conv_A):.2%})")

if (conv_B - conv_A) >= mde_absolute:
    print("\n✅ Текущий эффект ПРЕВЫШАЕТ MDE → тест имеет достаточную мощность")
else:
    print(" Текущий эффект НИЖЕ MDE → тест мог не обнаружить такой эффект")


РАСЧЕТ MDE (Minimal Detectable Effect)
Уровень значимости (α): 0.05
Мощность теста (1-β): 0.8
Текущий размер выборки: A=676, B=674

MDE (стандартизированный): 0.1525
MDE (в абсолютных п.п.): ±0.0398 (±3.98%)

Текущий наблюдаемый эффект: 0.0373 (3.73%)
 Текущий эффект НИЖЕ MDE → тест мог не обнаружить такой эффект


In [12]:
# Создаем итоговую таблицу для экспорта в Power BI
results_df = pd.DataFrame({
    'metric': [
        'users_A', 'users_B',
        'purchases_A', 'purchases_B',
        'conversion_A', 'conversion_B',
        'conversion_diff_abs', 'conversion_lift',
        'p_value', 'z_statistic',
        'ci_lower_95', 'ci_upper_95',
        'significant_05', 'mde_absolute',
        'alpha', 'power'
    ],
    'value': [
        users_A, users_B,
        purchases_A, purchases_B,
        conv_A, conv_B,
        conv_B - conv_A, (conv_B - conv_A) / conv_A,
        p_value, z_stat,
        ci_lower, ci_upper,
        1 if p_value < 0.05 else 0,
        mde_absolute,
        alpha, power
    ],
    'formatted': [
        f"{int(users_A)}", f"{int(users_B)}",
        f"{int(purchases_A)}", f"{int(purchases_B)}",
        f"{conv_A:.2%}", f"{conv_B:.2%}",
        f"+{(conv_B - conv_A):.2%}", f"+{((conv_B - conv_A) / conv_A):.1%}",
        f"{p_value:.6f}", f"{z_stat:.4f}",
        f"{ci_lower:.2%}", f"{ci_upper:.2%}",
        "✅ Да" if p_value < 0.05 else "❌ Нет",
        f"±{mde_absolute:.2%}",
        f"{alpha:.0%}", f"{power:.0%}"
    ]
})

print("\n" + "=" * 50)
print("ИТОГОВАЯ СТАТИСТИКА ДЛЯ POWER BI")
print("=" * 50)
print(results_df[['metric', 'formatted']].to_string(index=False))


ИТОГОВАЯ СТАТИСТИКА ДЛЯ POWER BI
             metric formatted
            users_A       676
            users_B       674
        purchases_A        37
        purchases_B        62
       conversion_A     5.47%
       conversion_B     9.20%
conversion_diff_abs    +3.73%
    conversion_lift    +68.1%
            p_value  0.008654
        z_statistic   -2.6254
        ci_lower_95     0.92%
        ci_upper_95     6.51%
     significant_05      ✅ Да
       mde_absolute    ±3.98%
              alpha        5%
              power       80%


In [13]:
# Сохраняем в CSV
results_df.to_csv('ab_test_stats.csv', index=False, encoding='utf-8-sig')
print(" Файл 'ab_test_stats.csv' сохранён для импорта в Power BI")
print(f" Путь к файлу: ab_test_stats.csv")

# Покажем содержимое файла для проверки
print("Содержимое файла (первые 5 строк):")
print(results_df.head().to_string(index=False))

 Файл 'ab_test_stats.csv' сохранён для импорта в Power BI
 Путь к файлу: ab_test_stats.csv
Содержимое файла (первые 5 строк):
      metric      value formatted
     users_A 676.000000       676
     users_B 674.000000       674
 purchases_A  37.000000        37
 purchases_B  62.000000        62
conversion_A   0.054734     5.47%
